# Modul 5: Detekcja obiektow YOLO — percepcja robota

W tym notebooku zaimplementujemy **pipeline percepcji** robota humanoidalnego Unitree G1, oparty na detekcji obiektow YOLO.

### Sensor wizyjny G1
Robot G1 wyposazony jest w kamere **Intel RealSense D435i**, ktora dostarcza:
- **Obraz RGB** — 640x480 @ 30 FPS
- **Mape glebokosci** — do 10m zasiegu
- **IMU** — akcelerometr + zyroskop

### Modele detekcji (porownanie z modulu 5)
| Model | Czas (Jetson Orin) | Opis |
|-------|-------------------|------|
| **YOLO26** | 10-20ms | Szybki, stale klasy (COCO) |
| **YOLOE** | 20-40ms | Open-vocabulary (prompt tekstowy) |
| **GroundingDINO** | 50-100ms | Najwyzsza dokladnosc |
| **SAM 2** | 30-80ms | Segmentacja masek pikseli |

W tym cwiczeniu uzyjemy **YOLO11** (ultralytics) — lekkiego modelu odpowiedniego do uruchomienia w Google Colab.

In [ ]:
!pip install ultralytics numpy matplotlib opencv-python-headless -q

## Detekcja YOLO na przykladowym obrazie

Zaladujemy pretrenowany model YOLO11n (nano — najlzejszy wariant) i uruchomimy detekcje na przykladowym obrazie.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np

# Zaladuj model YOLO11n (maly model — szybki download)
model = YOLO("yolo11n.pt")
print(f"Model zaladowany: {model.model_name}")

# Uruchom detekcje na przykladowym obrazie
results = model("https://ultralytics.com/images/bus.jpg")

# Wizualizacja wynikow
plt.figure(figsize=(12, 8))
plt.imshow(results[0].plot())
plt.axis('off')
plt.title('YOLO detekcja')
plt.show()

print(f"Wykryto {len(results[0].boxes)} obiektow")

## Analiza wynikow detekcji

Kazdy wykryty obiekt ma:
- **Klase** — nazwa obiektu (np. 'person', 'car', 'bus')
- **Pewnosc (confidence)** — wartosc 0-1 okreslajaca jak pewny jest model
- **Bounding box** — prostokat [x1, y1, x2, y2] otaczajacy obiekt

In [ ]:
# Parsowanie wynikow detekcji
result = results[0]

print(f"{'Klasa':<20} {'Pewnosc':>8}  {'Bbox (x1, y1, x2, y2)'}")
print("-" * 65)

detections = []
for box in result.boxes:
    cls_id = int(box.cls)
    cls_name = result.names[cls_id]
    confidence = float(box.conf)
    bbox = box.xyxy[0].tolist()  # [x1, y1, x2, y2]
    
    detections.append({
        "class": cls_name,
        "confidence": confidence,
        "bbox": bbox
    })
    
    bbox_str = f"[{bbox[0]:.0f}, {bbox[1]:.0f}, {bbox[2]:.0f}, {bbox[3]:.0f}]"
    print(f"  {cls_name:<18} {confidence:>7.2f}  {bbox_str}")

print(f"\nLaczna liczba detekcji: {len(detections)}")

# Statystyki klas
from collections import Counter
class_counts = Counter(d["class"] for d in detections)
print("\nStatystyki klas:")
for cls, count in class_counts.most_common():
    print(f"  {cls}: {count}")

## Detekcja na obrazie z perspektywy robota

Robot G1 widzi swiat z perspektywy kamery zamontowanej na glowie (ok. 1.2m wysokosci).
Ponizej symulujemy obraz z perspektywy robota — generujemy syntetyczna scene z obiektami na podlodze.

In [ ]:
import cv2

# Generujemy syntetyczny obraz — scene wnetrza z perspektywy robota
# (w rzeczywistosci uzylibysmy strumienia z RealSense D435i)
img_height, img_width = 480, 640
robot_view = np.zeros((img_height, img_width, 3), dtype=np.uint8)

# Tlo — podloga i sciana
robot_view[300:, :] = [180, 160, 140]  # podloga (jasny braz)
robot_view[:300, :] = [220, 215, 210]  # sciana (jasny szary)

# Obiekty na scenie
# Kubek (pomaranczowy prostokat)
cv2.rectangle(robot_view, (200, 280), (240, 320), (0, 140, 255), -1)
cv2.rectangle(robot_view, (200, 280), (240, 320), (0, 100, 200), 2)

# Butelka (zielony prostokat)
cv2.rectangle(robot_view, (400, 250), (430, 330), (0, 180, 0), -1)
cv2.rectangle(robot_view, (400, 250), (430, 330), (0, 120, 0), 2)

# Pilka (czerwone kolo)
cv2.circle(robot_view, (520, 350), 30, (0, 0, 220), -1)
cv2.circle(robot_view, (520, 350), 30, (0, 0, 160), 2)

# Krzeslo (brazowy ksztalt)
cv2.rectangle(robot_view, (80, 200), (160, 350), (60, 80, 120), -1)
cv2.rectangle(robot_view, (80, 200), (160, 350), (40, 60, 100), 2)

# Zapisz obraz tymczasowo
cv2.imwrite("robot_view.jpg", robot_view)

# Uruchom YOLO na syntetycznym obrazie
results_robot = model("robot_view.jpg")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Oryginalny obraz
axes[0].imshow(cv2.cvtColor(robot_view, cv2.COLOR_BGR2RGB))
axes[0].set_title('Widok z kamery robota (syntetyczny)', fontsize=12)
axes[0].axis('off')

# Wynik detekcji
axes[1].imshow(results_robot[0].plot())
axes[1].set_title('Detekcja YOLO', fontsize=12)
axes[1].axis('off')

plt.suptitle('Percepcja robota G1 — detekcja obiektow', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Wykryto {len(results_robot[0].boxes)} obiektow na obrazie robota")
print("\nUwaga: Na syntetycznym obrazie YOLO moze nie wykryc obiektow,")
print("poniewaz model jest trenowany na realistycznych zdjeciach.")
print("W rzeczywistosci uzylibysmy kamery RealSense D435i z prawdziwym strumieniem wideo.")

## Od 2D do 3D — fuzja z mapa glebokosci

Detekcja YOLO daje nam **bounding box w 2D** (piksele). Aby robot mogl nawigowac do obiektu, potrzebujemy **pozycji 3D** w przestrzeni.

Kamera RealSense D435i dostarcza **mape glebokosci** — dla kazdego piksela znamy odleglosc od kamery. Dzieki temu mozemy wykonac **deprojection** — przeliczenie z 2D + depth na 3D.

### Wzor deprojekcji
Majac:
- `(u, v)` — srodek bounding boxa w pikselach
- `depth` — glebokosc w metrach (z mapy glebokosci)
- `(fx, fy, cx, cy)` — parametry intrinsic kamery

Pozycja 3D w ukladzie kamery:
```
x = (u - cx) * depth / fx
y = (v - cy) * depth / fy
z = depth
```

In [ ]:
# Symulacja deprojekcji 2D -> 3D
# Parametry intrinsic kamery RealSense D435i (typowe wartosci)
fx, fy = 615.0, 615.0  # ogniskowa w pikselach
cx_cam, cy_cam = 320.0, 240.0  # srodek optyczny

def deproject_2d_to_3d(u, v, depth, fx, fy, cx, cy):
    """Przelicz pozycje 2D + glebokosc na pozycje 3D w ukladzie kamery."""
    x_3d = (u - cx) * depth / fx
    y_3d = (v - cy) * depth / fy
    z_3d = depth
    return x_3d, y_3d, z_3d

# Przykladowe detekcje z glebokoscia
example_detections = [
    {"class": "cup", "bbox": [200, 280, 240, 320], "depth": 1.5},
    {"class": "bottle", "bbox": [400, 250, 430, 330], "depth": 2.0},
    {"class": "ball", "bbox": [490, 320, 550, 380], "depth": 2.5},
    {"class": "chair", "bbox": [80, 200, 160, 350], "depth": 1.2},
]

print("Deprojekcja 2D -> 3D (uklad kamery)")
print("=" * 65)
print(f"{'Obiekt':<10} {'Bbox center':>14} {'Depth':>7} {'Poz. 3D (x, y, z)':>25}")
print("-" * 65)

positions_3d = []
for det in example_detections:
    bbox = det["bbox"]
    # Srodek bounding boxa
    u = (bbox[0] + bbox[2]) / 2
    v = (bbox[1] + bbox[3]) / 2
    depth = det["depth"]
    
    x_3d, y_3d, z_3d = deproject_2d_to_3d(u, v, depth, fx, fy, cx_cam, cy_cam)
    positions_3d.append((det["class"], x_3d, y_3d, z_3d))
    
    print(f"  {det['class']:<8} ({u:>5.0f}, {v:>5.0f})  {depth:>5.1f}m  "
          f"x={x_3d:>6.2f}m, y={y_3d:>6.2f}m, z={z_3d:>5.2f}m")

# Wizualizacja pozycji 3D (widok z gory)
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
colors = ['orange', 'green', 'red', 'brown']
for (name, x, y, z), color in zip(positions_3d, colors):
    ax.scatter(x, z, s=200, c=color, zorder=5)
    ax.annotate(f"{name}\n({x:.2f}, {z:.2f})", (x, z), 
                textcoords="offset points", xytext=(10, 10), fontsize=10)

# Robot na pozycji (0, 0)
ax.scatter(0, 0, s=300, c='blue', marker='^', zorder=5)
ax.annotate('Robot G1', (0, 0), textcoords="offset points", xytext=(10, -15), 
            fontsize=11, fontweight='bold')

ax.set_xlabel('X [m] (lewo/prawo)', fontsize=12)
ax.set_ylabel('Z [m] (odleglosc od kamery)', fontsize=12)
ax.set_title('Mapa obiektow — widok z gory (uklad kamery)', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 2)
ax.set_ylim(-0.5, 3.5)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Pipeline percepcji G1

Pelny pipeline percepcji na robocie G1 transformuje dane z kamery w pozycje 3D obiektow w ukladzie swiata:

```
RealSense D435i
  |                  
  |--- Obraz RGB ---------> YOLO detekcja -----> Bbox 2D + klasa
  |                                                    |
  |--- Mapa glebokosci ---> Odczyt depth(cx, cy) -----+
                                                       |
                                                       v
                                              Deprojekcja 2D->3D
                                              (uklad kamery)
                                                       |
                                                       v
                                            TF: camera -> base_link
                                              (transformacja sztywna)
                                                       |
                                                       v
                                            TF: base_link -> odom
                                              (z odometrii robota)
                                                       |
                                                       v
                                          Pozycja 3D w ukladzie swiata
                                              (x, y, z) [metry]
```

### Lancuch transformacji (TF)
1. **Camera frame** — uklad kamery (z = do przodu, x = w prawo, y = w dol)
2. **Base link** — srodek robota na poziomie podlogi
3. **Odom frame** — uklad odometrii (startowa pozycja robota)
4. **Map frame** — globalny uklad swiata (z SLAM lub nawigacji)

Na Jetson Orin caly pipeline (YOLO + depth + TF) dziala w **15-25ms**, co pozwala na detekcje w czasie rzeczywistym przy 30 FPS.

## Nastepne kroki

W **Module 6 (Nawigacja)** wykorzystamy wyniki detekcji z tego modulu do:

1. **Nawigacji do wykrytego obiektu** — robot G1 planuje sciezke do pozycji 3D obiektu
2. **Omijania przeszkod** — bounding boxy przeszkod sa uzywane do aktualizacji mapy kosztow
3. **Chwytania obiektow** — precyzyjna pozycja 3D pozwala na planowanie ruchu manipulatora

### Integracja z polityka RL
- Percepcja (Modul 5) dostarcza **cel** — pozycje obiektu
- Nawigacja (Modul 6) planuje **sciezke** do celu
- Polityka chodu (Modul 3-4) realizuje **ruch** wzdluz sciezki

```
Percepcja (YOLO + Depth)  -->  Pozycja 3D celu
                                      |
                                      v
                              Planer sciezki (A*, RRT)
                                      |
                                      v
                           Polityka RL (PPO) --> Sterowanie stawami
                                      |
                                      v
                                Robot G1 idzie do celu
```